# Testing/Looking at Weather Station Data

In [ ]:
#importing file correctly
import pandas as pd
import matplotlib.pyplot as plt

path = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/VIC/organised_weather_obs/HD01D_Data_086361_9999999910708904.txt"

df = pd.read_csv(
    path,
    sep=",",      # it’s comma-separated
    header=0      # first line is the header
)

In [ ]:
pd.read_csv(vic_stndet, sep=None, engine="python").columns.tolist()

In [ ]:
#check columns
df.columns.tolist()

In [ ]:
year_col   = [c for c in df.columns if "year" in c.lower() and "local" not in c.lower()][0]
month_col  = [c for c in df.columns if c == "MM"][0]
day_col    = [c for c in df.columns if c == "DD"][0]
hour_col   = [c for c in df.columns if c == "HH24"][0]
minute_col = [c for c in df.columns if "mi format in local time" in c.lower()][0]

year_col, month_col, day_col, hour_col, minute_col

In [ ]:
#building timestamp
df["timestamp"] = pd.to_datetime(
    df[year_col].astype(str).str.zfill(4) + "-" +
    df[month_col].astype(str).str.zfill(2) + "-" +
    df[day_col].astype(str).str.zfill(2) + " " +
    df[hour_col].astype(str).str.zfill(2) + ":" +
    df[minute_col].astype(str).str.zfill(2),
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

In [ ]:
#identify temperature column
[col for col in df.columns if "air temperature" in col.lower()]
temp_col = "Air Temperature in degrees Celsius"

In [ ]:
#filtering last 2 months of data
cutoff = df["timestamp"].max() - pd.DateOffset(months=2)
df_recent = df[df["timestamp"] >= cutoff]

#checking rows
len(df_recent)

## Reclassifying lat and lon to more notable name for title

In [ ]:
def classify_area(lat, lon):
    if lat < -38.2:
        return "South Coast / Bass Coast"
    if lat < -37.9:
        return "Mornington Peninsula"
    if lat < -37.6:
        return "Melbourne Metro"
    if lat < -37.3:
        return "Yarra Ranges / Dandenongs"
    return "Regional Victoria"

In [ ]:
area = classify_area(lat, lon)

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(
    df_recent["timestamp"],
    df_recent[temp_col],
    linewidth=1
)

plt.xlabel("Time")
plt.ylabel("Temperature (°C)")
plt.title(
    f"{area}\nTemperature over the past 2 months\n({clean_name})",
    loc="center"
)
plt.tight_layout()
plt.savefig(
    "/home/565/pv3484/aus_substation_electricity/figures/temperature_obs/temperature_basscoast_2months.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

# Using metadata to find location and name of this weather station

In [ ]:
import os

folder = os.path.dirname(path)
[x for x in os.listdir(folder) if "StnDet" in x]

In [ ]:
#loading correct StnDet file
stn_path = os.path.join(folder, "HD01D_StnDet_9999999910708904.txt")

stn = pd.read_csv(stn_path, header=None)

In [ ]:
stn.head(10)

In [ ]:
#renamining columns and assigning proper names
stn.columns = [
    "label", "st", "network", "name", "start", "end",
    "lat", "lon", "method", "state",
    "code1", "code2", "code3", "code4",
    "val1", "val2", "val3", "val4",
    "flag1", "flag2", "flag3", "flag4"
]

In [ ]:
#removing any whitespace in column names
stn["name"] = stn["name"].str.strip()
stn["lat"]  = stn["lat"].astype(float)
stn["lon"]  = stn["lon"].astype(float)

In [ ]:
#converting station ID to integer
stn["st"] = stn["st"].astype(int)

In [ ]:
#filtering for station
row = stn[stn["st"] == 86361]

In [ ]:
#creating variables (name, lat, lon) for plotting
name = row["name"].iloc[0]
lat  = row["lat"].iloc[0]
lon  = row["lon"].iloc[0]

print(name, lat, lon)

In [ ]:
#Capitalises only the first letter
clean_name = name.capitalize()

In [ ]:
repr(name), repr(area)

In [ ]:
repr(clean_name), repr(area)